# A/B Evaluation Analysis: Agentic Documentation Impact

## Purpose

This notebook analyzes the results of an A/B evaluation comparing **baseline** (no agentic docs)
vs **treatment** (with agentic docs) conditions for coding tasks across two repositories:

- `cert-manager-operator`
- `multiarch-tuning-operator`

Runs are stored in an MLflow experiment named **`agentic-docs-ab-eval`**.

---

## Methodology

| Step | Detail |
|------|--------|
| Condition | Baseline = no docs; Treatment = agentic docs injected at task time |
| Scoring | LLM-as-judge on 5 dimensions aggregated into **OES** (0–1) |
| Pairing | Runs matched by `tags.task_id` for paired statistical tests |
| Tests | Wilcoxon signed-rank (non-parametric, paired), Cohen's d, 95% bootstrap CI |

## OES Formula

```
OES = 0.30 * (task_success - 1) / 4
    + 0.25 * (code_correctness - 1) / 4
    + 0.20 * (convention_adherence - 1) / 4
    + 0.15 * hallucination_check
    + 0.10 * efficiency
```

## Decision Criteria

| Threshold | Interpretation |
|-----------|----------------|
| IDV > 0.10 | Meaningful improvement from agentic docs |
| IDV > 0.20 | Strong improvement |
| Pairwise win rate >= 0.55 | Treatment consistently preferred by LLM judge |


In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import wilcoxon, pearsonr
from IPython.display import display

import mlflow
import mlflow.entities
from mlflow.tracking import MlflowClient

# Suppress noisy runtime warnings from scipy/numpy on small samples
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ── Plotting defaults ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'sans-serif',
})

print('All imports loaded successfully.')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration — edit these constants if your setup differs
# ─────────────────────────────────────────────────────────────────────────────

EXPERIMENT_NAME = 'agentic-docs-ab-eval'

# MLflow tracking URI.
# Option A (file-based mlruns directory, as specified):
#   TRACKING_URI = os.path.join(os.path.abspath(''), 'mlruns')
# Option B (SQLite backend, used by eval-baseline.yaml / eval-treatment.yaml):
#   TRACKING_URI = 'sqlite:///' + os.path.join(os.path.abspath(''), 'mlflow.db')
#
# The notebook auto-detects which backend is present.
NOTEBOOK_DIR = os.path.abspath('')   # Jupyter sets CWD = notebook directory

_mlruns_path   = os.path.join(NOTEBOOK_DIR, 'mlruns')
_sqlite_path   = os.path.join(NOTEBOOK_DIR, 'mlflow.db')

if os.path.isdir(_mlruns_path):
    TRACKING_URI = _mlruns_path
elif os.path.isfile(_sqlite_path):
    TRACKING_URI = f'sqlite:///{_sqlite_path}'
else:
    # Default to mlruns (as specified in the task) even if absent
    TRACKING_URI = _mlruns_path

print(f'Notebook directory : {NOTEBOOK_DIR}')
print(f'MLflow tracking URI: {TRACKING_URI}')

# ── OES weight scheme (must sum to 1.0) ──────────────────────────────────────
# Weights reflect the relative importance of each dimension for coding tasks.
# task_success is the primary signal; efficiency is a softer bonus.
OES_WEIGHTS = {
    'task_success':         0.30,   # Normalized 1-5 -> 0-1
    'code_correctness':     0.25,   # Normalized 1-5 -> 0-1
    'convention_adherence': 0.20,   # Normalized 1-5 -> 0-1
    'hallucination_check':  0.15,   # Already 0/1 binary
    'efficiency':           0.10,   # Already 0/1 binary
}
assert abs(sum(OES_WEIGHTS.values()) - 1.0) < 1e-9, 'OES weights must sum to 1.0'

# ── Decision thresholds ───────────────────────────────────────────────────────
IDV_MEANINGFUL  = 0.10   # IDV > 0.10 -> meaningful lift
IDV_STRONG      = 0.20   # IDV > 0.20 -> strong lift
WIN_RATE_TARGET = 0.55   # Pairwise win rate target (head-to-head vs baseline)

# ── Stratification labels (used as fallbacks if tags are sparse) ──────────────
CATEGORIES   = ['api-change', 'test-writing', 'observability', 'security', 'architecture']
COMPLEXITIES = ['low', 'medium', 'high']
REPOS        = ['cert-manager-operator', 'multiarch-tuning-operator']

# ── Docs expected to be consulted (for ROI analysis) ─────────────────────────
EXPECTED_DOCS = [
    'AGENTS.md',
    'agentic/DESIGN.md',
    'agentic/DEVELOPMENT.md',
    'agentic/TESTING.md',
    'agentic/RELIABILITY.md',
    'agentic/SECURITY.md',
    'agentic/design-docs/components/trustmanager-controller.md',
    'agentic/design-docs/components/istiocsr-controller.md',
    'agentic/design-docs/components/optional-informer.md',
    'agentic/design-docs/components/pod-placement-webhook.md',
    'agentic/design-docs/components/ppo-controller.md',
    'agentic/design-docs/components/image-inspection-cache.md',
    'agentic/domain/concepts/singleton-pattern.md',
    'agentic/domain/concepts/execution-modes.md',
]

# ── Bootstrap settings ────────────────────────────────────────────────────────
N_BOOTSTRAP = 1_000
RNG_SEED    = 42
rng = np.random.default_rng(RNG_SEED)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load all runs from MLflow and split into baseline / treatment DataFrames
# ─────────────────────────────────────────────────────────────────────────────

mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(
        f"Experiment '{EXPERIMENT_NAME}' not found at '{TRACKING_URI}'.\n"
        'Check that TRACKING_URI is correct and that evaluation runs have been logged.'
    )

print(f'Experiment name  : {experiment.name}')
print(f'Experiment ID    : {experiment.experiment_id}')
print(f'Lifecycle stage  : {experiment.lifecycle_stage}')

# Pull all finished (active) runs; DELETED runs are excluded
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string='',
    run_view_type=mlflow.entities.ViewType.ACTIVE_ONLY,
)

if runs_df.empty:
    print('\n[WARNING] No runs found. The analysis cells will produce NaN results.')
    print('Run the eval harness first to populate run data.')

print(f'\nTotal runs loaded : {len(runs_df)}')
if len(runs_df):
    print(f'Columns           : {sorted(runs_df.columns.tolist())}')

# ── Derive condition column ───────────────────────────────────────────────────
# MLflow stores tags as "tags.<key>" and params as "params.<key>".
# We prefer the tag; fall back to the param if the tag is absent.
def _coalesce(df: pd.DataFrame, *cols: str) -> pd.Series:
    'Return the first non-null value across *cols* for each row.'
    result = pd.Series([None] * len(df), index=df.index, dtype=object)
    for col in cols:
        if col in df.columns:
            result = result.where(result.notna(), df[col])
    return result

runs_df['condition'] = (
    _coalesce(runs_df, 'tags.condition', 'params.condition')
    .astype(str)
    .str.strip()
    .str.lower()
    .replace('nan', pd.NA)
)

# ── Coerce metric columns to numeric ─────────────────────────────────────────
# LLM judges may emit string representations ("True"/"False", "1.0", etc.)
METRIC_COLS = [
    'metrics.task_success',
    'metrics.code_correctness',
    'metrics.convention_adherence',
    'metrics.hallucination_check',
    'metrics.efficiency',
    'metrics.consulted_docs_coverage',
    'metrics.consulted_docs_count',
    'metrics.pairwise_win',
]
for col in METRIC_COLS:
    if col in runs_df.columns:
        runs_df[col] = pd.to_numeric(runs_df[col], errors='coerce')

# ── Split conditions ──────────────────────────────────────────────────────────
baseline_df  = runs_df[runs_df['condition'] == 'baseline'].copy().reset_index(drop=True)
treatment_df = runs_df[runs_df['condition'] == 'treatment'].copy().reset_index(drop=True)

print(f'\nBaseline  runs : {len(baseline_df)}')
print(f'Treatment runs : {len(treatment_df)}')

if len(baseline_df) == 0 or len(treatment_df) == 0:
    print('\n[WARNING] One or both conditions have zero runs.')
    print('Subsequent cells will produce NaN for any metric requiring both conditions.')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# OES (Overall Effectiveness Score) computation
#
# OES = SUM( weight_i * normalized_metric_i )
#
# Normalization:
#   - 1-5 scale metrics -> (x - 1) / 4  so the range maps to [0, 1]
#   - Binary metrics (0/1) are used directly
#
# Missing value strategy:
#   - Impute with column median (conservative; acknowledges data gaps without
#     discarding the run entirely or biasing in either direction)
#   - After normalization, clip to [0, 1] to guard against out-of-range values
#     that could arise from logging errors or unexpected judge output
# ─────────────────────────────────────────────────────────────────────────────

def compute_oes(df: pd.DataFrame) -> pd.Series:
    'Compute OES for each row in *df*. Returns a Series aligned to df.index.'

    def safe_get(metric: str, default: float) -> pd.Series:
        col = f'metrics.{metric}'
        if col in df.columns:
            s = pd.to_numeric(df[col], errors='coerce')
            # Impute missing values with the column median; fall back to default
            med = s.median()
            return s.fillna(med if pd.notna(med) else default)
        return pd.Series([default] * len(df), index=df.index, dtype=float)

    # Normalize 1-5 scores to [0, 1]
    task_success         = ((safe_get('task_success',         3.0) - 1) / 4).clip(0, 1)
    code_correctness     = ((safe_get('code_correctness',     3.0) - 1) / 4).clip(0, 1)
    convention_adherence = ((safe_get('convention_adherence', 3.0) - 1) / 4).clip(0, 1)

    # Binary metrics — impute missing with 0 (pessimistic; avoids inflating OES)
    hallucination_check  = safe_get('hallucination_check', 0.0).clip(0, 1)
    efficiency           = safe_get('efficiency',          0.0).clip(0, 1)

    oes = (
          OES_WEIGHTS['task_success']         * task_success
        + OES_WEIGHTS['code_correctness']     * code_correctness
        + OES_WEIGHTS['convention_adherence'] * convention_adherence
        + OES_WEIGHTS['hallucination_check']  * hallucination_check
        + OES_WEIGHTS['efficiency']           * efficiency
    )
    return oes


baseline_df['OES']  = compute_oes(baseline_df)
treatment_df['OES'] = compute_oes(treatment_df)

print('OES computed for both conditions.')
print()
b_oes = baseline_df['OES']
t_oes = treatment_df['OES']
print(f'Baseline  OES  n={len(b_oes):3d}  mean={b_oes.mean():.4f}  std={b_oes.std():.4f}  '
      f'min={b_oes.min():.4f}  max={b_oes.max():.4f}')
print(f'Treatment OES  n={len(t_oes):3d}  mean={t_oes.mean():.4f}  std={t_oes.std():.4f}  '
      f'min={t_oes.min():.4f}  max={t_oes.max():.4f}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Per-run summary: mean +/- SD per condition across all key metrics
# ─────────────────────────────────────────────────────────────────────────────

DISPLAY_METRICS = {
    'metrics.task_success':         'Task Success (1-5)',
    'metrics.code_correctness':     'Code Correctness (1-5)',
    'metrics.convention_adherence': 'Convention Adherence (1-5)',
    'metrics.hallucination_check':  'Hallucination Check (0/1)',
    'metrics.efficiency':           'Efficiency (0/1)',
    'OES':                          'OES (composite, 0-1)',
}

rows = []
for col, label in DISPLAY_METRICS.items():
    b_vals = pd.to_numeric(
        baseline_df[col]  if col in baseline_df.columns  else pd.Series(dtype=float),
        errors='coerce'
    ).dropna()
    t_vals = pd.to_numeric(
        treatment_df[col] if col in treatment_df.columns else pd.Series(dtype=float),
        errors='coerce'
    ).dropna()

    delta = (t_vals.mean() - b_vals.mean()) if (len(b_vals) and len(t_vals)) else float('nan')

    rows.append({
        'Metric':          label,
        'Baseline mean':   round(b_vals.mean(), 4) if len(b_vals) else float('nan'),
        'Baseline SD':     round(b_vals.std(),  4) if len(b_vals) else float('nan'),
        'Baseline n':      len(b_vals),
        'Treatment mean':  round(t_vals.mean(), 4) if len(t_vals) else float('nan'),
        'Treatment SD':    round(t_vals.std(),  4) if len(t_vals) else float('nan'),
        'Treatment n':     len(t_vals),
        'Delta (T - B)':   round(delta, 4) if pd.notna(delta) else float('nan'),
    })

summary_df = pd.DataFrame(rows).set_index('Metric')
print('Per-condition metric summary (mean +/- SD):')
print()
display(summary_df.style.format({
    'Baseline mean':  '{:.4f}',
    'Baseline SD':    '{:.4f}',
    'Treatment mean': '{:.4f}',
    'Treatment SD':   '{:.4f}',
    'Delta (T - B)':  '{:+.4f}',
}).background_gradient(subset=['Delta (T - B)'], cmap='RdYlGn', vmin=-0.3, vmax=0.3))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# IDV (Incremental Documentation Value)
#
# IDV = mean(OES_treatment) - mean(OES_baseline)
#
# This is the primary scalar summary of the A/B evaluation.
# Thresholds are set to distinguish noise from meaningful gains:
#   IDV > 0.10 -> meaningful lift (worth the doc-maintenance cost)
#   IDV > 0.20 -> strong lift (clear recommendation to adopt agentic docs)
#
# Pairwise win rate: fraction of head-to-head comparisons (same task_id) where
# the treatment run was scored higher by the LLM judge. Target >= 0.55.
# ─────────────────────────────────────────────────────────────────────────────

mean_b = baseline_df['OES'].mean()
mean_t = treatment_df['OES'].mean()
idv    = mean_t - mean_b

print('=' * 55)
print('  IDV (Incremental Documentation Value)')
print('=' * 55)
print(f'  Mean OES baseline  : {mean_b:+.4f}')
print(f'  Mean OES treatment : {mean_t:+.4f}')
print(f'  IDV                : {idv:+.4f}')
print()
if pd.isna(idv):
    print('  [INFO] Cannot compute IDV — one or both conditions are empty.')
elif idv > IDV_STRONG:
    print(f'  STRONG improvement (IDV > {IDV_STRONG}): agentic docs deliver clear value.')
elif idv > IDV_MEANINGFUL:
    print(f'  MEANINGFUL improvement (IDV > {IDV_MEANINGFUL}): agentic docs add value.')
elif idv > 0:
    print(f'  MODEST improvement (0 < IDV < {IDV_MEANINGFUL}): marginal; '
          'consider cost vs benefit.')
else:
    print(f'  NO improvement (IDV <= 0): agentic docs did not lift performance '
          'on this evaluation set.')
print('=' * 55)

# ── Pairwise win rate ─────────────────────────────────────────────────────────
print()
print('Pairwise win rate analysis')
print('-' * 45)
if 'metrics.pairwise_win' in treatment_df.columns:
    pw = pd.to_numeric(treatment_df['metrics.pairwise_win'], errors='coerce').dropna()
    if len(pw) > 0:
        win_rate  = pw.mean()
        win_count = (pw == 1.0).sum()
        tie_count = (pw == 0.5).sum()
        los_count = (pw == 0.0).sum()
        print(f'  n comparisons : {len(pw)}')
        print(f'  Wins  (1.0)   : {win_count}  ({win_count/len(pw)*100:.1f}%)')
        print(f'  Ties  (0.5)   : {tie_count}  ({tie_count/len(pw)*100:.1f}%)')
        print(f'  Losses (0.0)  : {los_count}  ({los_count/len(pw)*100:.1f}%)')
        print(f'  Win rate      : {win_rate:.3f}  (target >= {WIN_RATE_TARGET})')
        print()
        if win_rate >= WIN_RATE_TARGET:
            print(f'  Treatment consistently preferred by LLM judge (win rate >= {WIN_RATE_TARGET}).')
        else:
            print(f'  Treatment NOT consistently preferred (win rate < {WIN_RATE_TARGET}).')
    else:
        print('  [INFO] pairwise_win column present but all values are NaN.')
        win_rate = float('nan')
else:
    print('  [INFO] metrics.pairwise_win not logged — pairwise analysis skipped.')
    win_rate = float('nan')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Statistical tests (one row per metric)
#
# 1. Wilcoxon signed-rank test (scipy.stats.wilcoxon)
#    - Non-parametric, paired — appropriate for ordinal LLM judge scores
#    - Paired by task_id; any task_id not present in both conditions is excluded
#    - Two-sided test; p < 0.05 => reject null that distributions are identical
#
# 2. Cohen's d (pooled-SD effect size)
#    - d < 0.2  => negligible effect
#    - 0.2-0.5  => small
#    - 0.5-0.8  => medium
#    - > 0.8    => large
#
# 3. 95% Bootstrap CI on mean difference (1000 resamples)
#    - Resamples paired differences to preserve within-task correlation
#    - Percentile method: [2.5th, 97.5th] of resampled means
#    - CI entirely above 0 provides additional evidence of a positive effect
# ─────────────────────────────────────────────────────────────────────────────

STAT_METRICS = [
    'metrics.task_success',
    'metrics.code_correctness',
    'metrics.convention_adherence',
    'metrics.hallucination_check',
    'metrics.efficiency',
    'OES',
]


def cohen_d(treatment: np.ndarray, baseline: np.ndarray) -> float:
    'Pooled-SD Cohen d (positive = treatment > baseline).'
    n_t, n_b = len(treatment), len(baseline)
    if n_t < 2 or n_b < 2:
        return float('nan')
    var_t = np.var(treatment, ddof=1)
    var_b = np.var(baseline,  ddof=1)
    pooled = ((n_t - 1) * var_t + (n_b - 1) * var_b) / (n_t + n_b - 2)
    return (np.mean(treatment) - np.mean(baseline)) / np.sqrt(pooled) if pooled > 0 else 0.0


def effect_label(d: float) -> str:
    ad = abs(d)
    if ad > 0.8:  return 'large'
    if ad > 0.5:  return 'medium'
    if ad > 0.2:  return 'small'
    return 'negligible'


def bootstrap_ci(
    treatment: np.ndarray,
    baseline:  np.ndarray,
    n_boot:    int   = N_BOOTSTRAP,
    alpha:     float = 0.05,
) -> tuple:
    'Paired 95% percentile bootstrap CI on (mean_treatment - mean_baseline).'
    if len(treatment) == len(baseline) and len(treatment) > 0:
        diffs = treatment - baseline
        # Resample paired differences (preserves within-task correlation)
        boots = rng.choice(diffs, size=(n_boot, len(diffs)), replace=True).mean(axis=1)
    else:
        # Unpaired fallback: independently resample each group
        boots_t = rng.choice(treatment, size=(n_boot, len(treatment)), replace=True).mean(axis=1)
        boots_b = rng.choice(baseline,  size=(n_boot, len(baseline)),  replace=True).mean(axis=1)
        boots = boots_t - boots_b
    return float(np.percentile(boots, 100 * alpha / 2)), float(np.percentile(boots, 100 * (1 - alpha / 2)))


def align_by_task(b_df: pd.DataFrame, t_df: pd.DataFrame) -> tuple:
    'Inner-join baseline and treatment on tags.task_id for paired comparisons.'
    task_col = 'tags.task_id'
    if task_col in b_df.columns and task_col in t_df.columns:
        b_keyed = b_df.dropna(subset=[task_col]).set_index(task_col)
        t_keyed = t_df.dropna(subset=[task_col]).set_index(task_col)
        common  = b_keyed.index.intersection(t_keyed.index)
        if len(common):
            return b_keyed.loc[common].reset_index(), t_keyed.loc[common].reset_index()
    # Fallback: positional pairing (truncate to shorter side)
    n = min(len(b_df), len(t_df))
    print(f'[INFO] No common task_id found; using positional pairing (n={n}).')
    return b_df.iloc[:n].copy().reset_index(drop=True), t_df.iloc[:n].copy().reset_index(drop=True)


b_aligned, t_aligned = align_by_task(baseline_df, treatment_df)
print(f'Paired observations for statistical tests: {len(b_aligned)}')
print()

stat_rows = []
for col in STAT_METRICS:
    b_raw = b_aligned[col] if col in b_aligned.columns else pd.Series(dtype=float)
    t_raw = t_aligned[col] if col in t_aligned.columns else pd.Series(dtype=float)

    b_s = pd.to_numeric(b_raw, errors='coerce')
    t_s = pd.to_numeric(t_raw, errors='coerce')

    mask  = b_s.notna() & t_s.notna()
    b_arr = b_s[mask].to_numpy()
    t_arr = t_s[mask].to_numpy()
    n     = len(b_arr)

    if n < 3:
        stat_rows.append({
            'Metric':      col.replace('metrics.', ''),
            'n pairs':     n,
            'Mean Delta':  float('nan'),
            'Wilcoxon p':  float('nan'),
            'Significant': 'n/a (too few)',
            "Cohen's d":   float('nan'),
            'Effect size': 'n/a',
            'CI 95% low':  float('nan'),
            'CI 95% high': float('nan'),
        })
        continue

    mean_delta = float(t_arr.mean() - b_arr.mean())

    # Wilcoxon: requires at least one non-zero difference
    diffs = t_arr - b_arr
    try:
        if np.all(diffs == 0):
            # All ties: no evidence of difference, p-value = 1
            w_p = 1.0
        else:
            _, w_p = wilcoxon(t_arr, b_arr, alternative='two-sided')
    except Exception as exc:
        print(f'  [WARN] Wilcoxon failed for {col}: {exc}')
        w_p = float('nan')

    d      = cohen_d(t_arr, b_arr)
    ci_lo, ci_hi = bootstrap_ci(t_arr, b_arr)

    stat_rows.append({
        'Metric':      col.replace('metrics.', ''),
        'n pairs':     n,
        'Mean Delta':  round(mean_delta, 4),
        'Wilcoxon p':  round(w_p, 4) if pd.notna(w_p) else float('nan'),
        'Significant': ('Yes (p<0.05)' if (pd.notna(w_p) and w_p < 0.05) else 'No'),
        "Cohen's d":   round(d, 3)  if pd.notna(d)   else float('nan'),
        'Effect size': effect_label(d) if pd.notna(d) else 'n/a',
        'CI 95% low':  round(ci_lo, 4),
        'CI 95% high': round(ci_hi, 4),
    })

stat_df = pd.DataFrame(stat_rows).set_index('Metric')

print('Statistical test results (treatment vs baseline):')
print()
print('Interpretation guide:')
print("  Cohen's d > 0.2 = small, > 0.5 = medium, > 0.8 = large effect")
print("  Wilcoxon p < 0.05 = statistically significant at 95% confidence")
print("  Bootstrap CI entirely above 0 => positive effect with 95% confidence")
print()
display(stat_df)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Per-category breakdown
#
# Group by tags.category, compute mean OES per condition, delta, and
# Wilcoxon p-value (where sufficient paired observations exist).
#
# This reveals whether certain task types benefit more from agentic docs,
# e.g. complex architecture tasks vs. simpler test-writing tasks.
# ─────────────────────────────────────────────────────────────────────────────

def per_group_breakdown(
    b_df:      pd.DataFrame,
    t_df:      pd.DataFrame,
    group_col: str,
    groups:    list,
    label:     str,
) -> pd.DataFrame:
    'Compute per-group OES stats. Uses positional pairing within each group.'
    rows = []
    for grp in groups:
        b_grp = b_df[b_df.get(group_col, pd.Series(dtype=str)) == grp]['OES'].dropna()
        t_grp = t_df[t_df.get(group_col, pd.Series(dtype=str)) == grp]['OES'].dropna()

        b_mean = b_grp.mean() if len(b_grp) else float('nan')
        t_mean = t_grp.mean() if len(t_grp) else float('nan')
        delta  = t_mean - b_mean if (pd.notna(t_mean) and pd.notna(b_mean)) else float('nan')

        n_min = min(len(b_grp), len(t_grp))
        if n_min >= 3:
            try:
                b_arr = b_grp.iloc[:n_min].to_numpy()
                t_arr = t_grp.iloc[:n_min].to_numpy()
                diffs = t_arr - b_arr
                w_p = 1.0 if np.all(diffs == 0) else wilcoxon(t_arr, b_arr)[1]
            except Exception:
                w_p = float('nan')
        else:
            w_p = float('nan')

        rows.append({
            label:          grp,
            'n baseline':   len(b_grp),
            'n treatment':  len(t_grp),
            'OES baseline': round(b_mean, 4) if pd.notna(b_mean) else float('nan'),
            'OES treatment':round(t_mean, 4) if pd.notna(t_mean) else float('nan'),
            'Delta OES':    round(delta,  4) if pd.notna(delta)  else float('nan'),
            'Wilcoxon p':   round(w_p,    4) if pd.notna(w_p)   else float('nan'),
            'Significant':  'Yes' if (pd.notna(w_p) and w_p < 0.05) else 'No',
        })
    return pd.DataFrame(rows).set_index(label)


cat_col = 'tags.category'
# Discover categories present in data; fall back to CATEGORIES constant
data_cats = sorted(set(
    list(baseline_df[cat_col].dropna().unique() if cat_col in baseline_df.columns else []) +
    list(treatment_df[cat_col].dropna().unique() if cat_col in treatment_df.columns else [])
)) or CATEGORIES

category_df = per_group_breakdown(baseline_df, treatment_df, cat_col, data_cats, 'Category')
print('OES by task category:')
print()
print('Higher Delta OES for a category => agentic docs especially helpful for that task type.')
print()
display(category_df.style.format({
    'OES baseline':  '{:.4f}',
    'OES treatment': '{:.4f}',
    'Delta OES':     '{:+.4f}',
    'Wilcoxon p':    '{:.4f}',
}).background_gradient(subset=['Delta OES'], cmap='RdYlGn', vmin=-0.3, vmax=0.3))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Per-complexity breakdown
#
# Hypothesis: higher-complexity tasks should benefit more from agentic docs
# because they involve more components, conventions, and domain knowledge.
# ─────────────────────────────────────────────────────────────────────────────

cplx_col = 'tags.complexity'

# Preserve logical order (low -> medium -> high)
data_cplx = []
for c in COMPLEXITIES:
    in_b = (c in baseline_df[cplx_col].values  if cplx_col in baseline_df.columns  else False)
    in_t = (c in treatment_df[cplx_col].values if cplx_col in treatment_df.columns else False)
    if in_b or in_t:
        data_cplx.append(c)

if not data_cplx:
    # Fall back to all complexity levels if no tagged data
    data_cplx = COMPLEXITIES

complexity_df = per_group_breakdown(baseline_df, treatment_df, cplx_col, data_cplx, 'Complexity')
print('OES by task complexity:')
print()
print('Hypothesis: High complexity tasks show larger Delta OES (more benefit from docs).')
print()
display(complexity_df.style.format({
    'OES baseline':  '{:.4f}',
    'OES treatment': '{:.4f}',
    'Delta OES':     '{:+.4f}',
    'Wilcoxon p':    '{:.4f}',
}).background_gradient(subset=['Delta OES'], cmap='RdYlGn', vmin=-0.3, vmax=0.3))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Per-repo breakdown
#
# Compare OES lift separately for cert-manager-operator vs
# multiarch-tuning-operator. The doc quality scores (agentic_docs_quality_score)
# are treatment-only params and are surfaced here for context.
# ─────────────────────────────────────────────────────────────────────────────

repo_col = 'tags.repo'

data_repos = sorted(set(
    list(baseline_df[repo_col].dropna().unique() if repo_col in baseline_df.columns else []) +
    list(treatment_df[repo_col].dropna().unique() if repo_col in treatment_df.columns else [])
)) or REPOS

repo_df = per_group_breakdown(baseline_df, treatment_df, repo_col, data_repos, 'Repository')

# Attach doc quality scores (treatment-only params)
REPO_SCORE_PARAMS = {
    'cert-manager-operator':     'params.agentic_docs_quality_score_cm',
    'multiarch-tuning-operator': 'params.agentic_docs_quality_score_mto',
}
for repo_label, score_param in REPO_SCORE_PARAMS.items():
    if repo_label in repo_df.index and score_param in treatment_df.columns:
        scores = pd.to_numeric(treatment_df[score_param], errors='coerce').dropna()
        repo_df.loc[repo_label, 'Doc quality score'] = (
            round(scores.mean(), 1) if len(scores) else float('nan')
        )

print('OES by repository:')
print()
print('Doc quality score = mean agentic_docs_quality_score param (treatment runs only).')
print('A high score with low Delta OES may indicate docs are good but not fully utilized.')
print()
display(repo_df.style.format({
    'OES baseline':    '{:.4f}',
    'OES treatment':   '{:.4f}',
    'Delta OES':       '{:+.4f}',
    'Wilcoxon p':      '{:.4f}',
    'Doc quality score': '{:.1f}',
}).background_gradient(subset=['Delta OES'], cmap='RdYlGn', vmin=-0.3, vmax=0.3))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Doc ROI coefficients
#
# For each document in EXPECTED_DOCS:
#   consultation_rate = fraction of treatment runs where the doc was consulted
#   impact            = Pearson r(was_doc_consulted, task_success)
#   roi               = impact * consultation_rate
#
# Higher ROI => the document is frequently consulted AND its consultation
# correlates with better task outcomes => high value per consultation.
#
# Data source: per-doc columns "metrics.doc_consulted_<docname>" (preferred)
# or "metrics.consulted_docs_coverage" as a proxy (flagged in output).
# ─────────────────────────────────────────────────────────────────────────────

task_success_t = pd.to_numeric(
    treatment_df.get('metrics.task_success', pd.Series(dtype=float)),
    errors='coerce'
)


def _doc_col_name(doc: str) -> str:
    'Derive expected MLflow column name for per-doc consultation flag.'
    safe = doc.replace('.', '_').replace('/', '_').replace('-', '_')
    return f'metrics.doc_consulted_{safe}'


roi_rows = []
for doc in EXPECTED_DOCS:
    col_name  = _doc_col_name(doc)
    is_proxy  = False

    if col_name in treatment_df.columns:
        # Per-doc binary flag is present
        consulted = pd.to_numeric(treatment_df[col_name], errors='coerce').fillna(0)
    else:
        # Fall back to consulted_docs_coverage as a continuous proxy.
        # All docs share the same proxy value, so ROI ranks are driven purely
        # by coverage correlation — treat results with caution.
        cov_col   = 'metrics.consulted_docs_coverage'
        consulted = pd.to_numeric(
            treatment_df.get(cov_col, pd.Series(dtype=float)), errors='coerce'
        )
        is_proxy = True

    rate      = consulted.mean() if consulted.notna().any() else float('nan')
    valid     = consulted.notna() & task_success_t.notna()
    n_valid   = valid.sum()

    if n_valid >= 3:
        corr, pval = pearsonr(consulted[valid], task_success_t[valid])
    else:
        corr, pval = float('nan'), float('nan')

    roi = (corr * rate) if (pd.notna(corr) and pd.notna(rate)) else float('nan')

    roi_rows.append({
        'Document':           doc,
        'Consultation rate':  round(rate, 3)  if pd.notna(rate) else float('nan'),
        'Impact r':           round(corr, 3)  if pd.notna(corr) else float('nan'),
        'p-value':            round(pval, 4)  if pd.notna(pval) else float('nan'),
        'ROI score':          round(roi,  4)  if pd.notna(roi)  else float('nan'),
        'n valid':            int(n_valid),
        'Proxy?':             'Yes (coverage)' if is_proxy else 'No',
    })

roi_df = (
    pd.DataFrame(roi_rows)
    .set_index('Document')
    .sort_values('ROI score', ascending=False)
)

print('Doc ROI coefficients (treatment runs only):')
print()
print('Formula  : ROI = Impact (Pearson r with task_success) x Consultation rate')
print('Interpret: Higher ROI => more valuable per consultation.')
print('           Positive r => consulting doc correlates with better outcomes.')
print('           Proxy = Yes means per-doc flag absent; coverage used as stand-in.')
print()
display(roi_df.style.format({
    'Consultation rate': '{:.3f}',
    'Impact r':          '{:.3f}',
    'p-value':           '{:.4f}',
    'ROI score':         '{:.4f}',
}).background_gradient(subset=['ROI score'], cmap='Greens'))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualizations
#
# Panel 1: OES box + strip plots (distribution by condition)
# Panel 2: Forest plot — 95% bootstrap CI on mean difference per metric
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('A/B Evaluation: Agentic Docs Impact', fontsize=15, fontweight='bold', y=1.01)

PALETTE = {'Baseline': '#6baed6', 'Treatment': '#2ca25f'}
DARK_PAL = {'Baseline': '#2171b5', 'Treatment': '#006d2c'}

# ── Panel 1: OES box / strip plot ────────────────────────────────────────────
ax1 = axes[0]

oes_long = pd.concat([
    baseline_df[['OES']].assign(Condition='Baseline'),
    treatment_df[['OES']].assign(Condition='Treatment'),
], ignore_index=True)

if not oes_long.empty:
    sns.boxplot(
        data=oes_long, x='Condition', y='OES',
        palette=PALETTE, width=0.45, linewidth=1.5,
        flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.5},
        ax=ax1,
    )
    sns.stripplot(
        data=oes_long, x='Condition', y='OES',
        palette=DARK_PAL, size=4, alpha=0.4, jitter=True, ax=ax1,
    )
    # Annotate means
    for i, (cond, color) in enumerate(zip(['Baseline', 'Treatment'], ['#2171b5', '#006d2c'])):
        grp = oes_long[oes_long['Condition'] == cond]['OES']
        if len(grp):
            m = grp.mean()
            ax1.axhline(m, color=color, linestyle='--', alpha=0.7, linewidth=1)
            ax1.text(i + 0.25, m + 0.01, f'mean={m:.3f}', color=color,
                     fontsize=9, ha='left', va='bottom')
else:
    ax1.text(0.5, 0.5, 'No data available', ha='center', va='center',
             transform=ax1.transAxes, fontsize=12, color='grey')

ax1.set_title('OES Distribution by Condition')
ax1.set_ylabel('Overall Effectiveness Score (0-1)')
ax1.set_xlabel('')
ax1.set_ylim(-0.05, 1.10)

# ── Panel 2: Forest plot ──────────────────────────────────────────────────────
ax2 = axes[1]

forest_data = stat_df.reset_index().copy()
forest_data = forest_data[forest_data['Mean Delta'].notna()].reset_index(drop=True)

ax2.axvline(0, color='black', linewidth=1.0, linestyle='-', zorder=1)
ax2.axvline(IDV_MEANINGFUL, color='#fc8d59', linewidth=1.0, linestyle='--',
            alpha=0.7, zorder=1, label=f'IDV meaningful ({IDV_MEANINGFUL})')

if len(forest_data):
    y_labels = forest_data['Metric'].tolist()
    y_pos    = list(range(len(y_labels)))
    colors   = [
        '#2ca25f' if row['Significant'] == 'Yes (p<0.05)' else '#969696'
        for _, row in forest_data.iterrows()
    ]

    for i, (_, row) in enumerate(forest_data.iterrows()):
        lo    = row.get('CI 95% low',  float('nan'))
        hi    = row.get('CI 95% high', float('nan'))
        mean_ = row['Mean Delta']
        col   = colors[i]

        if pd.notna(lo) and pd.notna(hi):
            ax2.plot([lo, hi], [i, i], color=col, linewidth=2, zorder=2)
            ax2.plot([lo, lo], [i - 0.1, i + 0.1], color=col, linewidth=2)
            ax2.plot([hi, hi], [i - 0.1, i + 0.1], color=col, linewidth=2)
        ax2.plot(mean_, i, 'o', color=col, markersize=8, zorder=3)

        if row['Significant'] == 'Yes (p<0.05)':
            ax2.annotate('*', xy=(hi if pd.notna(hi) else mean_, i),
                         xytext=(5, 0), textcoords='offset points',
                         fontsize=14, color='#2ca25f', va='center')

    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(y_labels, fontsize=10)
else:
    ax2.text(0.5, 0.5, 'No data for forest plot', ha='center', va='center',
             transform=ax2.transAxes, fontsize=12, color='grey')

# Legend
sig_patch   = mpatches.Patch(color='#2ca25f', label='Significant (p<0.05)')
insig_patch = mpatches.Patch(color='#969696', label='Not significant')
ax2.legend(handles=[sig_patch, insig_patch], fontsize=9, loc='lower right')
ax2.set_xlabel('Mean difference (treatment - baseline)')
ax2.set_title('Forest Plot: 95% Bootstrap CI on Mean Delta')

plt.tight_layout()
out_path = 'ab_eval_analysis.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {out_path}')


# A/B Evaluation: Conclusions

---

## Summary of Findings

Run this cell after executing all analysis cells above. Key outputs to reference:

| Signal | Value | Threshold | Pass? |
|--------|-------|-----------|-------|
| IDV    | `{idv:.3f}` (from Cell 7) | > 0.10 | — |
| Pairwise win rate | `{win_rate:.3f}` (from Cell 7) | >= 0.55 | — |
| OES Wilcoxon p | (from Cell 8) | < 0.05 | — |

*(Fill in actual values after running the notebook.)*

---

## Interpretation Guide

### Positive result (IDV > 0.10 AND win rate >= 0.55)
- Agentic docs deliver measurable, consistent improvement
- Recommend shipping docs alongside the eval harness for future tasks

### Mixed result (IDV > 0 but < 0.10, or win rate 0.50-0.55)
- Modest signal; check per-category and per-complexity breakdowns
- May indicate docs help for specific task types but not universally

### Null result (IDV <= 0 or win rate < 0.50)
- Docs did not improve LLM performance on this evaluation set
- Investigate: doc coverage (consulted_docs_coverage), doc quality scores,
  and whether high-complexity tasks show any positive signal

---

## Recommended Next Steps

1. **Examine category breakdown (Cell 9)**: If certain categories show large positive delta,
   consider targeted doc improvements for those areas.

2. **Check complexity gradient (Cell 10)**: Verify that high-complexity tasks benefit most
   from docs; this validates the doc-value hypothesis.

3. **Review Doc ROI (Cell 12)**: Prioritize maintaining high-ROI docs; consider removing or
   redesigning low-ROI docs that are rarely consulted.

4. **Re-run with more tasks**: Statistical power increases with sample size.
   Aim for n >= 30 per condition for reliable Wilcoxon results.

5. **Per-repo comparison (Cell 11)**: If one repo shows stronger lift, investigate whether
   its docs are more comprehensive (doc quality score) or better matched to task types.
